In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType,TimestampType,DataType
from pyspark.sql import functions as F


In [0]:
catalog_name='ecommerce'
#Define schema for the data file 

brand_schema=StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code', StringType(), True),
 ])


In [0]:
raw_data_path="/Volumes/ecommerce/source_data/raw/brands/*.csv"

df=spark.read\
    .option('header',True)\
    .option('delimeter',',')\
    .schema(brand_schema)\
    .csv(raw_data_path)

#add Metadata columns

df=df.withColumn('_source_file',F.col('_metadata.file_path'))\
    .withColumn('_ingest_ts',F.current_timestamp())

display(df.limit(10))

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema','true')\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")


In [0]:
#Category

category_schema=StructType([
    StructField('category_code',StringType(),False),
    StructField('category_name',StringType(),True),
])

cat_raw_path='/Volumes/ecommerce/source_data/raw/category/*.csv'

df_cat=spark.read\
    .option('header',True)\
    .option('delimeter',',')\
    .schema(category_schema)\
    .csv(cat_raw_path)
#add Metadata columns
df_cat=df_cat.withColumn('_source_file',F.col('_metadata.file_path'))\
             .withColumn('_ingest_ts',F.current_timestamp())

df_cat.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.bronze.brz_category")

In [0]:
# Product

product_schema=StructType([
    StructField('product_id',StringType(),False),
    StructField('sku',StringType(),True),
    StructField('category_code',StringType(),True),
    StructField('brand_code',StringType(),True),
    StructField('color',StringType(),True),
    StructField('size',StringType(),True),
    StructField('material',StringType(),True),
    StructField('weight_grams',StringType(),True),
    StructField('length_cm',StringType(),True),
    StructField('width_cm',FloatType(),True),
    StructField('height_cm',FloatType(),True),
    StructField('rating_count',IntegerType(),True),
])

pro_raw_path='/Volumes/ecommerce/source_data/raw/products/*.csv'

df_pro=spark.read\
    .option('header',True)\
    .option('delimeter',',')\
    .schema(product_schema)\
    .csv(pro_raw_path)
#add Metadata columns
df_pro=df_pro.withColumn('file_name',F.col('_metadata.file_path'))\
             .withColumn('ingest_time',F.current_timestamp())

#raw data is storing the delta table under in the bronze schema

df_pro.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.bronze.brz_products")

In [0]:
#Date

catalog_name='ecommerce'
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql import functions as F

raw_date_path='/Volumes/ecommerce/source_data/raw/date/*.csv'

date_schema=StructType([
    StructField('date',StringType(),False),
    StructField('year',IntegerType(),True),
    StructField('day_name',StringType(),True),
    StructField('quater',IntegerType(),True),
    StructField('week_of_year',IntegerType(),True),
    ])
df_date=spark.read\
    .option('header',True)\
        .option('delimeter',',')\
            .schema(date_schema)\
                .csv(raw_date_path)
df_date=df_date.withColumn('file_name',F.col('_metadata.file_path'))\
                .withColumn('ingest_time',F.current_timestamp())

df_date.write.format('delta')\
    .mode('overwrite')\
        .option('mergeSchema',True)\
            .saveAsTable(f"{catalog_name}.bronze.brz_date")

In [0]:
#Customers

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

raw_cus_path='/Volumes/ecommerce/source_data/raw/customers/*.csv'

df_cus=spark.read\
    .option('header',True)\
    .option('delimeter',',')\
    .schema(customers_schema)\
    .csv(raw_cus_path)
df_cus=df_cus.withColumn('file_name',F.col('_metadata.file_path'))\
    .withColumn('ingest_time',F.current_timestamp())

df_cus.write.format('delta')\
    .mode('overwrite')\
        .option('mergeSchema',True)\
        .saveAsTable(f"{catalog_name}.bronze.brz_customers")

